In [2]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Sequential Public Goods Game with Q-learning Agents\n",
    "\n",
    "This notebook simulates a simple sequential Public Goods Game (PGG) where multiple agents repeatedly decide whether to contribute to a shared public good or defect (free ride).\n",
    "\n",
    "Agents use independent Q-learning to learn policies mapping states to actions (contribute or defect) over episodes.\n",
    "\n",
    "## Setup\n",
    "- Grid-based groups (no spatial movement to keep it simple)\n",
    "- Groups of fixed size interact each round\n",
    "- Payoff depends on total group contributions multiplied by synergy factor\n",
    "- Q-learning updates based on received rewards\n",
    "\n",
    "## Goals\n",
    "- Observe if cooperation emerges over episodes\n",
    "- Track average contribution rates over time\n",
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "# Environment parameters\n",
    "NUM_AGENTS = 20          # total agents\n",
    "GROUP_SIZE = 5           # agents per group\n",
    "NUM_GROUPS = NUM_AGENTS // GROUP_SIZE\n",
    "EPISODES = 200           # training episodes\n",
    "STEPS_PER_EPISODE = 1    # one step per episode for simplicity\n",
    "\n",
    "# PGG parameters\n",
    "CONTRIBUTE_COST = 1.0     # cost to contribute\n",
    "SYNERGY_FACTOR = 1.5      # multiplied public good\n",
    "\n",
    "# Actions\n",
    "CONTRIBUTE = 1\n",
    "DEFECT = 0\n",
    "\n",
    "# Q-learning parameters\n",
    "ALPHA = 0.1      # learning rate\n",
    "GAMMA = 0.9      # discount factor\n",
    "EPSILON = 0.1    # exploration rate\n",
    "\n",
    "np.random.seed(42)  # reproducible"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Agent Class"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "class Agent:\n",
    "    def __init__(self, id):\n",
    "        self.id = id\n",
    "        # Simple state: last action (0 or 1), start with 0\n",
    "        self.state = 0\n",
    "        # Q-table: 2 states (last action), 2 actions (contribute or defect)\n",
    "        self.q_table = np.zeros((2, 2))\n",
    "\n",
    "    def choose_action(self):\n",
    "        # Epsilon-greedy\n",
    "        if np.random.rand() < EPSILON:\n",
    "            return np.random.choice([CONTRIBUTE, DEFECT])\n",
    "        else:\n",
    "            return np.argmax(self.q_table[self.state])\n",
    "\n",
    "    def update_q(self, action, reward, next_state):\n",
    "        predict = self.q_table[self.state, action]\n",
    "        target = reward + GAMMA * np.max(self.q_table[next_state])\n",
    "        self.q_table[self.state, action] += ALPHA * (target - predict)\n",
    "        self.state = next_state"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Environment Simulation Function"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "def run_episode(agents, groups):\n",
    "    actions = []\n",
    "    # Each agent chooses action\n",
    "    for agent in agents:\n",
    "        action = agent.choose_action()\n",
    "        actions.append(action)\n",
    "\n",
    "    # Compute rewards for each agent based on group contributions\n",
    "    rewards = np.zeros(len(agents))\n",
    "    for group in groups:\n",
    "        group_actions = [actions[i] for i in group]\n",
    "        total_contrib = sum(group_actions)\n",
    "        # Public good is synergy * total contribution\n",
    "        public_good = total_contrib * SYNERGY_FACTOR\n",
    "        # Each agent gets equal share\n",
    "        share = public_good / len(group)\n",
    "        # Assign rewards\n",
    "        for i in group:\n",
    "            cost = CONTRIBUTE_COST if actions[i] == CONTRIBUTE else 0\n",
    "            rewards[i] = share - cost\n",
    "\n",
    "    # Update Q tables\n",
    "    for i, agent in enumerate(agents):\n",
    "        # Next state = last action\n",
    "        next_state = actions[i]\n",
    "        agent.update_q(actions[i], rewards[i], next_state)\n",
    "\n",
    "    return actions, rewards"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Setup Agents and Groups"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "agents = [Agent(i) for i in range(NUM_AGENTS)]\n",
    "\n",
    "# Fixed groups: sequential slices of agents\n",
    "groups = [list(range(i * GROUP_SIZE, (i + 1) * GROUP_SIZE)) for i in range(NUM_GROUPS)]"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Run Training Loop"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "avg_contributions = []\n",
    "for ep in range(EPISODES):\n",
    "    actions, rewards = run_episode(agents, groups)\n",
    "    avg_contributions.append(np.mean(actions))\n",
    "    if (ep + 1) % 20 == 0:\n",
    "        print(f\"Episode {ep+1}: Average contribution {avg_contributions[-1]:.3f}\")"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Plot Average Contributions Over Episodes"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "plt.plot(avg_contributions)\n",
    "plt.xlabel('Episode')\n",
    "plt.ylabel('Average Contribution')\n",
    "plt.title('Cooperation Emergence in Sequential Public Goods Game')\n",
    "plt.show()"
   ],
   "execution_count": null,
   "outputs": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined


"# Sequential Public Goods Game with Q-learning Agents\n",
"\n",
"This notebook simulates a simple sequential Public Goods Game (PGG) where multiple agents repeatedly decide whether to contribute to a shared public good or defect (free ride).\n",
"\n",
"Agents use independent Q-learning to learn policies mapping states to actions (contribute or defect) over episodes.\n",
"\n",
"## Setup\n",
"- Grid-based groups (no spatial movement to keep it simple)\n",
"- Groups of fixed size interact each round\n",
"- Payoff depends on total group contributions multiplied by synergy factor\n",
"- Q-learning updates based on received rewards\n",
"\n",
"## Goals\n",
"- Observe if cooperation emerges over episodes\n",
"- Track average contribution rates over time\n",
"\n"
